<!-- # Visualisations

Loads saved results from `Results/` and produces publication-quality figures.
Each figure is saved to `Plots/` as a PDF (+ PNG) with metadata embedded in
a sidecar `_meta.json` file.

**Run order**
1. *Setup* — imports, style, helpers
2. *Q(k) diagnostics* — raw and interpolated self-energy
3. *Exciton & photon dispersion*
4. *Lower polariton dispersion*
5. *Hopfield coefficients*
6. *Detuning*
7. *Interaction strengths vs k* -->


In [ ]:
import sys
sys.path.insert(0, '.')

import json
import datetime
from dataclasses import replace
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, FixedLocator, MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import PchipInterpolator

from polaritons.io         import load_result, load_latest, list_results
from polaritons.dispersion import DispersionModel
from polaritons.many_body  import (
	hopfield_coefficients,
	polariton_interaction_strength,
	Pi0,
)
from polaritons.parameters import Params

# ---------------------------------------------------------------------------
# Matplotlib style
# ---------------------------------------------------------------------------
CMAP_NAME = "magma"
CMAP = plt.get_cmap(CMAP_NAME)
SHOW_M_PRIME_LABEL = True

plt.rcParams.update({
	'font.family'               : 'serif',
	'font.serif'                : ['DejaVu Serif'],
	'font.size'                 : 14,
	'axes.labelsize'            : 16,
	'axes.titlesize'            : 16,
	'xtick.labelsize'           : 14,
	'ytick.labelsize'           : 14,
	'legend.fontsize'           : 13,
	'figure.titlesize'          : 18,
	'axes.formatter.limits'     : (-99, 99),
	'axes.formatter.useoffset'  : False,
	'image.cmap'                : CMAP_NAME,
})

PLOTS_DIR = Path("Plots")
PLOTS_DIR.mkdir(exist_ok=True)
CM_INV_PER_M_INV = 0.01
CM_INV_LABEL = "cm⁻¹"
INTERACTION_UNIT_LABEL = "μeV μm²"
SUPERSCRIPT = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")

# ---------------------------------------------------------------------------
# Plot-boundary unit conversion helpers
# ---------------------------------------------------------------------------

def k_nat_to_cm(k_nat, p_nat):
	values = CM_INV_PER_M_INV * np.asarray(k_nat, dtype=float) / p_nat.L_unit
	return float(values) if values.ndim == 0 else values


def k_nat_to_cm_array(k_nat, p_nat):
	return np.asarray(k_nat_to_cm(k_nat, p_nat), dtype=float)


def k_cm_to_nat(k_cm, p_nat):
	values = np.asarray(k_cm, dtype=float) * p_nat.L_unit / CM_INV_PER_M_INV
	return float(values) if values.ndim == 0 else values


def energy_nat_to_eV(E_nat, p_nat):
	return np.asarray(E_nat) * p_nat.E_unit


def energy_nat_to_meV(E_nat, p_nat):
	return 1e3 * energy_nat_to_eV(E_nat, p_nat)


def interaction_nat_to_microev_um2(g_nat, p_nat):
	return 1e18 * np.asarray(g_nat) * p_nat.E_unit * p_nat.L_unit**2


def eta_color(index: int, total: int):
	denom = max(total - 1, 1)
	return CMAP(0.12 + 0.83 * index / denom)


def cavity_label(disorder_tuned: bool) -> str:
	return "disorder-tuned cavity" if disorder_tuned else "disorder-free cavity"


def cavity_slug(disorder_tuned: bool) -> str:
	return "disorder_tuned" if disorder_tuned else "disorder_free"


def superscript_int(value: int) -> str:
	return str(int(value)).translate(SUPERSCRIPT)


def format_number(value) -> str:
	value = float(value)
	if not np.isfinite(value):
		return ""
	if abs(value) < 5e-13:
		value = 0.0
	value = round(value, 5)
	if abs(value - round(value)) < 5e-12:
		return f"{int(round(value)):,}" if abs(value) >= 1000 else str(int(round(value)))
	text = f"{value:,.5f}" if abs(value) >= 1000 else f"{value:.5f}"
	return text.rstrip("0").rstrip(".")


def format_power_value(value) -> str:
	value = float(value)
	if not np.isfinite(value):
		return ""
	if abs(value) < 1e-12:
		return "0"
	sign = "−" if value < 0 else ""
	value = abs(value)
	exponent = int(np.floor(np.log10(value)))
	coefficient = value / 10**exponent
	return f"{sign}{format_number(coefficient)}×10{superscript_int(exponent)}"


def hide_axis_offset(ax):
	ax.xaxis.get_offset_text().set_visible(False)
	ax.yaxis.get_offset_text().set_visible(False)


def format_axis_numbers(ax, *, x=True, y=True):
	if x:
		ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_number(value)))
	if y:
		ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_number(value)))
	hide_axis_offset(ax)


def format_cm_ticks(ax, xmax, *, tick_count=5):
	if xmax <= 0 or not np.isfinite(xmax):
		return
	ax.xaxis.set_major_locator(MaxNLocator(nbins=tick_count - 1, min_n_ticks=2))
	ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_number(value)))
	hide_axis_offset(ax)


def format_power_cm_ticks(ax, xmax, *, tick_count=5):
	if xmax <= 0 or not np.isfinite(xmax):
		return
	ax.xaxis.set_major_locator(MaxNLocator(nbins=tick_count - 1, min_n_ticks=2))
	ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_power_value(value)))
	hide_axis_offset(ax)


def format_contour_axes(ax):
	ticks = [0, 10_000, 20_000, 30_000, 40_000]
	ax.xaxis.set_major_locator(FixedLocator(ticks))
	ax.yaxis.set_major_locator(FixedLocator(ticks))
	ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_number(value)))
	ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: format_number(value)))
	hide_axis_offset(ax)


def clean_number_slug(value):
	text = f"{float(value):.6g}"
	return text.replace("-", "m").replace("+", "").replace(".", "p")


def display_kernel_type(kernel_type):
	labels = {
		"gaussian"   : "Gaussian",
		"nongaussian": "Non-Gaussian",
		"ng"         : "Non-Gaussian",
	}
	return labels.get(str(kernel_type), str(kernel_type))



def build_result_slug(kernel_type, xi_m, m_prime):
	kernel_part = str(kernel_type).replace(" ", "_").replace("-", "_").lower()
	xi_part = "xi_independent" if xi_m is None else f"xi{clean_number_slug(1e9 * xi_m)}nm"
	return f"{kernel_part}_{xi_part}_mprime{clean_number_slug(m_prime)}"


def build_result_label(kernel_type, xi_m, m_prime):
	parts = [display_kernel_type(kernel_type)]
	if xi_m is not None:
		parts.append(f"ξ={1e9 * xi_m:g} nm")
	if SHOW_M_PRIME_LABEL:
		parts.append(f"m'={m_prime:g}")
	return ", ".join(parts)


def context_metadata(ctx):
	xi_value = None if ctx["xi_m"] is None else float(ctx["xi_m"])
	return {
		"source_stem" : ctx["q_stem"],
		"kernel_stem" : ctx["kernel_stem"],
		"kernel_type" : ctx["kernel_type"],
		"xi_independent": bool(ctx["xi_independent"]),
		"xi_m"        : xi_value,
		"m_prime"     : float(ctx["m_prime"]),
		"result_label": ctx["result_label_text"],
		"calculation_units": "natural",
		"sweep_schema": SWEEP_SCHEMA,
	}

# ---------------------------------------------------------------------------
# Figure-saving helper
# ---------------------------------------------------------------------------

def save_figure(fig, name: str, meta=None, dpi: int = 150) -> None:
	"""
	Save `fig` to Plots/<name>.pdf and Plots/<name>.png, and write metadata
	to Plots/<name>_meta.json.
	"""
	pdf_path  = PLOTS_DIR / f"{name}.pdf"
	png_path  = PLOTS_DIR / f"{name}.png"
	json_path = PLOTS_DIR / f"{name}_meta.json"

	fig.savefig(pdf_path, bbox_inches="tight")
	fig.savefig(png_path, bbox_inches="tight", dpi=dpi)

	record = {
		"saved_at" : datetime.datetime.now(datetime.timezone.utc).isoformat(),
		"pdf"      : str(pdf_path),
		"png"      : str(png_path),
		"cmap"     : CMAP_NAME,
		**(meta or {}),
	}
	with open(json_path, "w") as f:
		json.dump(record, f, indent=2, default=str)
	print(f"Saved  {pdf_path.name}")

print("Setup complete.")


## Load results

Choose which saved result set to visualise.  By default the most recently
saved Q result is loaded; change `Q_STEM` / `SWEEP_IDX` to pick a specific run.


In [ ]:
# Set Q_STEMS to a list of specific stems, or None to plot every current-schema Q result.
Q_STEMS = None
SHOW_M_PRIME_LABEL = True
SWEEP_SCHEMA = "gaussian_xi_plus_single_nongaussian_v1"

PARAM_FIELDS = (
	"E_bind", "E_gap_bare", "m_e", "m_h", "m_rest",
	"Omega", "m_prime", "n_refr", "N_qw", "D_0", "xi",
	"T", "concentration", "g_ex", "in_natural_units",
	"E_unit", "L_unit", "c", "hbar", "k_B", "M_eff",
)


def reconstruct_params(raw):
	kwargs = {name: raw[name] for name in PARAM_FIELDS if name in raw}
	params = Params(**kwargs)
	if not params.in_natural_units:
		raise ValueError("Q metadata must contain natural-unit Params. Regenerate results with Polarition Disorder.ipynb.")
	return params


def discover_q_stems():
	if Q_STEMS is not None:
		return list(Q_STEMS)
	metas = [
		meta for meta in list_results("Results/Q_results", prefix="Q")
		if meta.get("calculation_units") == "natural" and meta.get("sweep_schema") == SWEEP_SCHEMA
	]
	if not metas:
		raise FileNotFoundError(
			"No current-schema natural Q results found in Results/Q_results. "
			"Run Polarition Disorder.ipynb after this update."
		)
	return [meta["_stem"] for meta in metas]


def build_plot_context(q_stem_value):
	Q_data, meta = load_result("Results/Q_results", q_stem_value)
	if meta.get("calculation_units") != "natural" or meta.get("sweep_schema") != SWEEP_SCHEMA:
		raise ValueError(f"{q_stem_value} is not a current-schema natural result; regenerate it.")
	meta["_stem"] = q_stem_value
	raw = meta["params"]
	params = reconstruct_params(raw)
	eta_values = np.array(meta["eta_grid"])
	q_values = np.array(meta["q_picard"])
	kernel_name = meta["kernel_type"]
	xi_independent = bool(meta.get("xi_independent", False))
	xi_value = None if xi_independent else float(meta["xi_m"])
	m_prime_value = float(meta.get("m_prime", raw["m_prime"]))
	return {
		"Q_results"        : Q_data,
		"q_meta"           : meta,
		"q_stem"           : q_stem_value,
		"p_nat"            : params,
		"eta_grid"         : eta_values,
		"q_picard"         : q_values,
		"kernel_stem"      : meta.get("kernel_stem"),
		"kernel_type"      : kernel_name,
		"xi_independent"   : xi_independent,
		"xi_m"             : xi_value,
		"m_prime"          : m_prime_value,
		"result_slug_text" : build_result_slug(kernel_name, xi_value, m_prime_value),
		"result_label_text": build_result_label(kernel_name, xi_value, m_prime_value),
	}


def context_sort_key(ctx):
	kernel_rank = 0 if ctx["kernel_type"] in ("nongaussian", "ng") else 1
	xi_rank = -float("inf") if ctx["xi_m"] is None else float(ctx["xi_m"])
	return (kernel_rank, xi_rank, float(ctx["m_prime"]), str(ctx["q_stem"]))


def apply_context(ctx):
	for key, value in ctx.items():
		globals()[key] = value
	globals()["plot_context"] = ctx

	# Model, self-energy, and many-body calculations all use natural units.
	model_value = DispersionModel(p_nat, q_picard, eta_grid, Q_results)
	globals()["model"] = model_value

	# Plot grids are sampled in natural units and converted only for axis labels.
	k_domain_max_value = float(q_picard.max())
	positive_q = q_picard[q_picard > 0]
	if positive_q.size == 0:
		raise ValueError(f"Q grid for {q_stem} has no positive momentum samples.")
	first_positive_value = float(positive_q[0])
	k_disp_max = min(float(k_cm_to_nat(40_000.0, p_nat)), k_domain_max_value)
	if k_disp_max <= first_positive_value:
		k_disp_max = k_domain_max_value
	if k_disp_max <= first_positive_value:
		raise ValueError(f"Q grid for {q_stem} does not span more than one positive momentum sample.")

	globals().update({
		"k_domain_max"   : k_domain_max_value,
		"first_positive_k": first_positive_value,
		"k_zero"         : np.array([0.0]),
		"k_disp"         : np.linspace(first_positive_value, k_disp_max, 200),
		"k_wide"         : np.linspace(first_positive_value, k_domain_max_value, 500),
		"k_full_log"     : np.geomspace(first_positive_value, k_domain_max_value, 700),
	})
	globals().update({
		"k_disp_cm"    : k_nat_to_cm_array(k_disp, p_nat),
		"k_wide_cm"    : k_nat_to_cm_array(k_wide, p_nat),
		"k_full_log_cm": k_nat_to_cm_array(k_full_log, p_nat),
	})
	return model_value


plot_contexts = [build_plot_context(stem) for stem in discover_q_stems()]
plot_contexts.sort(key=context_sort_key)
apply_context(plot_contexts[0])

print(f"Loaded {len(plot_contexts)} Q result context(s):")
for ctx in plot_contexts:
	shape = ctx["Q_results"].shape
	print(f"  {ctx['q_stem']}: {ctx['result_label_text']}  shape={shape}  saved_at={ctx['q_meta'].get('saved_at')}")
print(f"Momentum domain of first context: 0 to {float(k_nat_to_cm(k_domain_max, p_nat)):.3g} {CM_INV_LABEL}")


In [ ]:
plot_eta_idx_base = [0, 5, 10, 15, 20]

for plot_context in plot_contexts:
	apply_context(plot_context)
	plot_eta_idx = [idx for idx in plot_eta_idx_base if idx < len(eta_grid)]
	q_positive = q_picard > 0.0
	q_plot_cm = k_nat_to_cm_array(q_picard[q_positive], p_nat)

	fig, axs = plt.subplots(1, 2, figsize=(18, 6))

	for idx in plot_eta_idx:
		clr = eta_color(idx, len(eta_grid))
		lbl = f"η={eta_grid[idx]:.2f}"
		axs[0].plot(q_plot_cm, Q_results[idx, q_positive].real,
					color=clr, label=f"{lbl} (Re)")
		axs[0].plot(q_plot_cm, Q_results[idx, q_positive].imag,
					color=clr, linestyle='--', label=f"{lbl} (Im)")

		k_nat_test = np.linspace(first_positive_k, 0.9 * k_domain_max, 300)
		k_test_cm = k_nat_to_cm_array(k_nat_test, p_nat)
		Q_interp  = model.Q(k_nat_test, eta_grid[idx])
		axs[1].plot(k_test_cm, np.real(Q_interp),
					color=clr, label=f"{lbl} (Re)")
		axs[1].plot(k_test_cm, np.imag(Q_interp),
					color=clr, linestyle='--', label=f"{lbl} (Im)")

	for ax, title in zip(axs, ["Raw Q(k)  (natural units)", "Interpolated Q(k)  (natural units)"]):
		ax.set_xlabel(f"k ({CM_INV_LABEL})")
		ax.set_ylabel("Q")
		ax.set_title(f"{title} [{result_label_text}]")
		ax.legend(ncol=2, fontsize=10)
		ax.grid(alpha=0.3)
		format_power_cm_ticks(ax, float(ax.get_xlim()[1]), tick_count=5)
		format_axis_numbers(ax, x=False, y=True)

	plt.tight_layout()
	save_figure(fig, f"Q_diagnostics_{result_slug_text}", meta={**context_metadata(plot_context), "eta_indices": plot_eta_idx})
	plt.show()
	plt.close(fig)


## Kernel contour K(q,k)

Downsampled from the saved Picard kernel mesh, normalized by its maximum value, and plotted on a linear color scale.


In [ ]:
# Kernel contour plot K(q,k), downsampled from the saved Picard mesh.
K_CONTOUR_POINTS = 500
K_CONTOUR_MAX_CM = 40_000.0
K_CONTOUR_TICKS = [0, 10_000, 20_000, 30_000, 40_000]
K_COLOR_TICKS = [0.0, 0.25, 0.50, 0.75, 1.0]

for plot_context in plot_contexts:
	apply_context(plot_context)
	if kernel_stem is None:
		print(f"Skipping K contour for {result_label_text}: no kernel_stem in Q metadata")
		continue

	K_mesh, k_meta = load_result("Results/integrand_meshes", kernel_stem)
	q_all = np.array(k_meta["q_picard"])
	q_window_max = float(k_cm_to_nat(K_CONTOUR_MAX_CM, p_nat))
	window_idx = np.flatnonzero(q_all <= q_window_max)
	if window_idx.size < 2:
		raise ValueError(f"K contour for {result_label_text} has fewer than two grid points below {K_CONTOUR_MAX_CM:g} {CM_INV_LABEL}.")

	stride = max(1, int(np.ceil(window_idx.size / K_CONTOUR_POINTS)))
	idx = np.unique(np.r_[window_idx[::stride], window_idx[-1]])
	K_small = np.asarray(K_mesh[np.ix_(idx, idx)], dtype=float)
	finite = K_small[np.isfinite(K_small)]
	if finite.size == 0:
		raise ValueError(f"K contour for {result_label_text} requires at least one finite kernel value.")
	K_max = float(np.nanmax(finite))
	if K_max <= 0.0:
		raise ValueError(f"K contour for {result_label_text} requires a positive kernel maximum.")
	K_plot = np.clip(K_small / K_max, 0.0, 1.0)
	q_kernel = q_all[idx]
	q_kernel_cm = k_nat_to_cm_array(q_kernel, p_nat)

	fig, ax = plt.subplots(figsize=(8, 6))
	mesh = ax.pcolormesh(q_kernel_cm, q_kernel_cm, K_plot,
						 shading="auto", cmap=CMAP_NAME, vmin=0.0, vmax=1.0)
	cbar = plt.colorbar(mesh, ax=ax, label="K(q,k) / max K")
	cbar.set_ticks(K_COLOR_TICKS)
	cbar.set_ticklabels([f"{value:.2f}" for value in K_COLOR_TICKS])
	ax.set_xlabel(f"k ({CM_INV_LABEL})")
	ax.set_ylabel(f"q ({CM_INV_LABEL})")
	ax.set_xlim(0.0, K_CONTOUR_MAX_CM)
	ax.set_ylim(0.0, K_CONTOUR_MAX_CM)
	ax.set_title(f"Disorder kernel K(q,k) [{result_label_text}]")
	format_contour_axes(ax)
	ax.grid(alpha=0.2)
	plt.tight_layout()
	save_figure(fig, f"K_contour_{result_slug_text}", meta={
		**context_metadata(plot_context),
		"downsample_stride": int(stride),
		"points": int(len(idx)),
		"norm": "max",
		"kernel_max": K_max,
		"k_max_cm_inv": K_CONTOUR_MAX_CM,
		"q_max_cm_inv": K_CONTOUR_MAX_CM,
	})
	plt.show()
	plt.close(fig)


## Exciton energy at k=0 vs disorder strength


In [ ]:
for plot_context in plot_contexts:
	apply_context(plot_context)
	E_ex_k0 = np.array([model.E_ex(np.array([0.0]), eta)[0] for eta in eta_grid])

	eta_dense = np.linspace(eta_grid[0], eta_grid[-1], 201)
	E_real_dense_eV = energy_nat_to_eV(PchipInterpolator(eta_grid, np.real(E_ex_k0))(eta_dense), p_nat)
	E_imag_dense_meV = energy_nat_to_meV(PchipInterpolator(eta_grid, np.imag(E_ex_k0))(eta_dense), p_nat)

	fig, ax = plt.subplots(figsize=(8, 6))
	ax_r = ax.twinx()

	real_color = CMAP(0.32)
	imag_color = CMAP(0.82)
	ax.plot(eta_dense, E_real_dense_eV, color=real_color, lw=1.5, label="Real")
	ax_r.plot(eta_dense, E_imag_dense_meV, color=imag_color, lw=1.5, label="Imaginary")

	ax.set_xlabel("Disorder strength η")
	ax.set_ylabel(r"Re[$E_x$(0)] (eV)", color=real_color)
	ax_r.set_ylabel(r"Im[$E_x$(0)] (meV)", color=imag_color)
	ax.tick_params(axis='y', colors=real_color)
	ax_r.tick_params(axis='y', colors=imag_color)
	ax.set_title(f"Exciton energy at k=0 vs η [{result_label_text}]")
	format_axis_numbers(ax)
	format_axis_numbers(ax_r, x=False, y=True)
	ax.grid(alpha=0.3)
	ax.legend(loc="upper left"); ax_r.legend(loc="upper right")
	plt.tight_layout()
	save_figure(fig, f"Eex_k0_vs_eta_{result_slug_text}", meta=context_metadata(plot_context))
	plt.show()
	plt.close(fig)


## Dispersion relations

Exciton and photon dispersion for a selected disorder value, and full
exciton / LP dispersions for all η.


In [ ]:
for plot_context in plot_contexts:
	apply_context(plot_context)
	eta_probe_idx = min(10, len(eta_grid) - 1)
	eta_probe = eta_grid[eta_probe_idx]
	k_cm = k_disp_cm

	E_ex_probe = model.E_ex(k_disp, eta_probe)
	E_ex_clean = model.E_ex(k_disp, 0.0)

	for tuned in (True, False):
		suffix = cavity_label(tuned)
		E_ph_use = model.E_ph(k_disp, eta_probe) if tuned else model.E_ph_untuned(k_disp)

		fig, axs = plt.subplots(1, 2, figsize=(18, 7))
		axs[0].plot(k_cm, energy_nat_to_eV(np.real(E_ex_probe), p_nat), color=CMAP(0.25), lw=2, label=f"Exciton η={eta_probe:.1f}")
		axs[0].plot(k_cm, energy_nat_to_eV(np.real(E_ex_clean), p_nat), color=CMAP(0.55), lw=2, label="Exciton η=0")
		axs[0].plot(k_cm, energy_nat_to_eV(E_ph_use, p_nat), color=CMAP(0.85), lw=2, label="Photon")
		axs[1].plot(k_cm, energy_nat_to_meV(np.imag(E_ex_probe), p_nat), color=CMAP(0.25), lw=2, label=f"Exciton η={eta_probe:.1f}")
		axs[1].plot(k_cm, energy_nat_to_meV(np.imag(E_ex_clean), p_nat), color=CMAP(0.55), lw=2, label="Exciton η=0")

		for ax in axs:
			ax.set_xlabel(f"k ({CM_INV_LABEL})")
			ax.legend(loc="lower right")
			ax.grid(alpha=0.3)
			format_cm_ticks(ax, float(k_cm.max()))
			format_axis_numbers(ax, x=False, y=True)
		axs[0].set_ylabel("E (eV)")
		axs[1].set_ylabel("E (meV)")
		axs[0].set_title(f"Dispersion -- Real  [{suffix}; {result_label_text}]")
		axs[1].set_title(f"Dispersion -- Imaginary  [{suffix}; {result_label_text}]")
		axs[0].text(0.04, 0.92, "(a)", transform=axs[0].transAxes, fontsize=22)
		axs[1].text(0.04, 0.92, "(b)", transform=axs[1].transAxes, fontsize=22)

		plt.tight_layout()
		save_figure(fig, f"dispersion_{result_slug_text}_{cavity_slug(tuned)}_cavity", meta={
			**context_metadata(plot_context),
			"eta_probe": float(eta_probe),
			"disorder_tuned": tuned,
		})
		plt.show()
		plt.close(fig)


## Lower polariton dispersion — all η


In [ ]:
legend_etas = [0.0, 0.5, 1.0, 1.5, 2.0]

for plot_context in plot_contexts:
	apply_context(plot_context)
	for tuned in (True, False):
		label = cavity_label(tuned)
		fig, axs = plt.subplots(1, 2, figsize=(18, 7))
		for ei, eta in enumerate(eta_grid):
			clr = eta_color(ei, len(eta_grid))
			lbl = f"η={eta:.1f}" if eta in legend_etas else "_nolegend_"
			E_lp = model.E_LP(k_disp, eta, disorder_tuned=tuned)
			axs[0].plot(k_disp_cm, energy_nat_to_eV(np.real(E_lp), p_nat), color=clr, label=lbl)
			axs[1].plot(k_disp_cm, energy_nat_to_meV(np.imag(E_lp), p_nat), color=clr, label=lbl)

		for ax in axs:
			ax.set_xlabel(f"k ({CM_INV_LABEL})")
			ax.legend(loc="lower right")
			ax.grid(alpha=0.3)
			format_cm_ticks(ax, float(k_disp_cm.max()))
			format_axis_numbers(ax, x=False, y=True)
		axs[0].set_ylabel("Re[E_LP] (eV)")
		axs[1].set_ylabel("Im[E_LP] (meV)")
		axs[0].set_title(f"LP Dispersion -- Real  [{label}; {result_label_text}]")
		axs[1].set_title(f"LP Dispersion -- Imaginary  [{label}; {result_label_text}]")
		axs[0].text(0.04, 0.95, "(a)", transform=axs[0].transAxes, fontsize=22)
		axs[1].text(0.04, 0.95, "(b)", transform=axs[1].transAxes, fontsize=22)

		plt.tight_layout()
		save_figure(fig, f"LP_dispersion_{result_slug_text}_{cavity_slug(tuned)}", meta={
			**context_metadata(plot_context),
			"disorder_tuned": tuned,
		})
		plt.show()
		plt.close(fig)


## Hopfield coefficients


In [ ]:
for plot_context in plot_contexts:
	apply_context(plot_context)
	for tuned in (True, False):
		fig, axs = plt.subplots(1, 2, figsize=(16, 6))
		for eta in legend_etas:
			ei = int(np.argmin(np.abs(eta_grid - eta)))
			clr = eta_color(ei, len(eta_grid))
			eta_value = float(eta_grid[ei])
			E_lp = model.E_LP(k_disp, eta_value, disorder_tuned=tuned)
			X_LP, C_LP = hopfield_coefficients(model, eta_value, k_disp, E_lp)
			axs[0].plot(k_disp_cm, np.abs(X_LP)**2, color=clr, label=f"η={eta_value:.1f}")
			axs[1].plot(k_disp_cm, np.abs(C_LP)**2, color=clr, label=f"η={eta_value:.1f}")

		for ax, title in zip(axs, ["Exciton fraction |X_LP|²", "Photon fraction |C_LP|²"]):
			ax.set_xlabel(f"k ({CM_INV_LABEL})")
			ax.set_ylabel("Hopfield coefficient²")
			ax.set_title(f"{title}  [{cavity_label(tuned)}; {result_label_text}]")
			ax.set_ylim(0, 1.05)
			ax.legend()
			ax.grid(alpha=0.3)
			format_cm_ticks(ax, float(k_disp_cm.max()))
			format_axis_numbers(ax, x=False, y=True)

		plt.tight_layout()
		save_figure(fig, f"hopfield_coefficients_{result_slug_text}_{cavity_slug(tuned)}", meta={
			**context_metadata(plot_context),
			"disorder_tuned": tuned,
		})
		plt.show()
		plt.close(fig)


## Detuning Δ(k, η) = E_ph − E_ex


In [ ]:
for plot_context in plot_contexts:
	apply_context(plot_context)
	for tuned in (True, False):
		fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
		for ei, eta in enumerate(eta_grid):
			clr = eta_color(ei, len(eta_grid))
			lbl = f"η={eta:.1f}" if eta in legend_etas else "_nolegend_"
			E_ph = model.E_ph(k_wide, eta) if tuned else model.E_ph_untuned(k_wide)
			delta = E_ph - model.E_ex(k_wide, eta)
			ax1.plot(k_wide_cm, energy_nat_to_meV(np.real(delta), p_nat), color=clr, label=lbl)
			ax2.plot(k_wide_cm, energy_nat_to_meV(np.imag(delta), p_nat), color=clr, label=lbl)

		for ax in (ax1, ax2):
			ax.axhline(0, color=CMAP(0.05), lw=0.8, ls='--')
			ax.set_xlabel(f"k ({CM_INV_LABEL})")
			ax.legend()
			ax.grid(alpha=0.3)
			format_power_cm_ticks(ax, float(k_wide_cm.max()), tick_count=4)
			format_axis_numbers(ax, x=False, y=True)

		ax1.set_ylabel(r"Re[$E_{ph}−E_{ex}$] (meV)")
		ax1.set_title(f"Real detuning vs momentum  [{cavity_label(tuned)}; {result_label_text}]")
		ax2.set_ylabel(r"Im[$E_{ph}−E_{ex}$] (meV)")
		ax2.set_title(f"Imaginary detuning vs momentum  [{cavity_label(tuned)}; {result_label_text}]")

		plt.tight_layout()
		save_figure(fig, f"detuning_{result_slug_text}_{cavity_slug(tuned)}", meta={
			**context_metadata(plot_context),
			"disorder_tuned": tuned,
		})
		plt.show()
		plt.close(fig)


## Polariton interaction strength vs k


In [ ]:
# Parameters for the many-body calculation
L_TERMS = 100
K_UPPER = 1.0    # natural momentum
N_K = 100_000
INTERACTION_INSET_MAX_CM = 5_000.0


def add_low_k_inset(ax, curves, *, x_max=INTERACTION_INSET_MAX_CM):
	if not curves:
		return
	available_max = max(float(np.nanmax(x)) for x, _, _ in curves if len(x) > 0)
	inset_max = min(float(x_max), available_max)
	if inset_max <= 0.0:
		return

	inset = inset_axes(ax, width="42%", height="42%", loc="upper right", borderpad=1.2)
	y_segments = []
	for x_values, y_values, color in curves:
		mask = x_values <= inset_max
		if np.count_nonzero(mask) < 2:
			mask = np.arange(len(x_values)) < min(20, len(x_values))
		inset.plot(x_values[mask], y_values[mask], color=color, alpha=0.9)
		y_segments.append(y_values[mask])

	y_all = np.concatenate([segment[np.isfinite(segment)] for segment in y_segments if segment.size])
	if y_all.size:
		y_min = min(0.0, float(np.nanmin(y_all)))
		y_max = float(np.nanmax(y_all))
		pad = 0.05 * max(y_max - y_min, abs(y_max), 1.0)
		inset.set_ylim(y_min, y_max + pad)
	inset.set_xlim(0.0, inset_max)
	inset.set_title("k≈0", fontsize=10)
	inset.grid(alpha=0.2)
	format_cm_ticks(inset, inset_max, tick_count=3)
	format_axis_numbers(inset, x=False, y=True)
	inset.tick_params(labelsize=9)


def plot_interaction_family(k_values: np.ndarray, *, log_k: bool, name_suffix: str, plot_context) -> None:
	x_values = k_nat_to_cm_array(k_values, p_nat)
	bubble_cache = {}

	def bubble_for(eta: float, disorder_tuned: bool) -> float:
		key = (float(eta), bool(disorder_tuned))
		if key not in bubble_cache:
			bubble_cache[key] = Pi0(
				model, eta,
				L_terms=L_TERMS,
				k_upper=K_UPPER,
				n_k=N_K,
				disorder_tuned=disorder_tuned,
			)
		return bubble_cache[key]

	for tuned in (True, False):
		for bare, g_label, y_label in [
			(True,  "bare",    f"g ({INTERACTION_UNIT_LABEL})"),
			(False, "screened", f"g' ({INTERACTION_UNIT_LABEL})"),
		]:
			fig, ax = plt.subplots(figsize=(8, 6))
			curves = []
			for ei, eta in enumerate(eta_grid):
				clr = eta_color(ei, len(eta_grid))
				lbl = f"η={eta:.1f}" if eta in legend_etas else "_nolegend_"
				g_bare = polariton_interaction_strength(
					model, eta, k_values,
					bare=True,
					L_terms=L_TERMS,
					k_upper=K_UPPER,
					n_k=N_K,
					disorder_tuned=tuned,
				)
				if bare:
					g = g_bare
				else:
					bubble = bubble_for(eta, tuned)
					g = g_bare * (1.0 - g_bare * bubble) / (1.0 - 2.0 * g_bare * bubble)
				y_values = interaction_nat_to_microev_um2(g, p_nat)
				curves.append((x_values, y_values, clr))
				ax.plot(x_values, y_values, color=clr, label=lbl, alpha=0.9)

			ax.set_xlabel(f"k ({CM_INV_LABEL})")
			ax.set_ylabel(y_label)
			ax.set_title(f"{g_label.capitalize()} interaction strength  [{cavity_label(tuned)}; {result_label_text}]")
			ax.legend()
			ax.set_ylim(0, None)
			ax.grid(alpha=0.3, which="both")
			if log_k:
				ax.set_xscale("log")
				ax.set_xlim(float(x_values[0]), float(x_values[-1]))
				format_axis_numbers(ax, x=False, y=True)
			else:
				format_cm_ticks(ax, float(x_values.max()))
				format_axis_numbers(ax, x=False, y=True)
			add_low_k_inset(ax, curves)

			plt.tight_layout()
			save_figure(fig, f"g_{g_label}_{result_slug_text}_{cavity_slug(tuned)}{name_suffix}",
						meta={
							**context_metadata(plot_context),
							"bare": bare,
							"disorder_tuned": tuned,
							"L_terms": L_TERMS,
							"k_upper_natural": K_UPPER,
							"n_k": N_K,
							"log_k": log_k,
							"k_min_cm_inv": float(x_values[0]),
							"k_max_cm_inv": float(x_values[-1]),
							"inset_max_cm_inv": INTERACTION_INSET_MAX_CM,
						})
			plt.show()
			plt.close(fig)


for plot_context in plot_contexts:
	apply_context(plot_context)
	plot_interaction_family(k_disp, log_k=False, name_suffix="", plot_context=plot_context)
	plot_interaction_family(k_full_log, log_k=True, name_suffix="_full_logk", plot_context=plot_context)


## Interaction strength at k=0 vs temperature

Exact k=0 values for bare and screened interactions across temperature.


In [ ]:
TEMPERATURES = np.linspace(5.0, 100.0, 20)  # K
K_ZERO = np.array([0.0])
TEMP_N_K = 30_000

for plot_context in plot_contexts:
	apply_context(plot_context)

	def model_at_temperature(T: float) -> DispersionModel:
		return DispersionModel(replace(p_nat, T=float(T)), q_picard, eta_grid, Q_results)

	for tuned in (True, False):
		fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
		for eta in legend_etas:
			ei = int(np.argmin(np.abs(eta_grid - eta)))
			eta_value = float(eta_grid[ei])
			clr = eta_color(ei, len(eta_grid))
			bare_vals = []
			screened_vals = []
			for T in TEMPERATURES:
				temp_model = model_at_temperature(float(T))
				bubble = Pi0(
					temp_model, eta_value,
					L_terms=L_TERMS,
					k_upper=K_UPPER,
					n_k=TEMP_N_K,
					disorder_tuned=tuned,
				)
				g_bare = polariton_interaction_strength(
					temp_model, eta_value, K_ZERO,
					bare=True,
					disorder_tuned=tuned,
				)[0]
				g_screened = g_bare * (1.0 - g_bare * bubble) / (1.0 - 2.0 * g_bare * bubble)
				bare_vals.append(g_bare)
				screened_vals.append(g_screened)

			axs[0].plot(TEMPERATURES, interaction_nat_to_microev_um2(np.asarray(bare_vals), p_nat), color=clr, label=f"η={eta_value:.1f}")
			axs[1].plot(TEMPERATURES, interaction_nat_to_microev_um2(np.asarray(screened_vals), p_nat), color=clr, label=f"η={eta_value:.1f}")

		for ax in axs:
			ax.set_xlabel("T (K)")
			ax.grid(alpha=0.3)
			ax.legend()
			format_axis_numbers(ax)
		axs[0].set_ylabel(f"g(0) ({INTERACTION_UNIT_LABEL})")
		axs[1].set_ylabel(f"g'(0) ({INTERACTION_UNIT_LABEL})")
		axs[0].set_title(f"Bare interaction at k=0  [{cavity_label(tuned)}; {result_label_text}]")
		axs[1].set_title(f"Screened interaction at k=0  [{cavity_label(tuned)}; {result_label_text}]")
		plt.tight_layout()
		save_figure(fig, f"g_k0_vs_temperature_{result_slug_text}_{cavity_slug(tuned)}", meta={
			**context_metadata(plot_context),
			"disorder_tuned": tuned,
			"temperatures_K": TEMPERATURES.tolist(),
			"L_terms": L_TERMS,
			"k_upper_natural": K_UPPER,
			"n_k": TEMP_N_K,
		})
		plt.show()
		plt.close(fig)
